# tda-witness demo

Topological Data Analysis from scratch. Point cloud in, Betti numbers out.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avacadobanana352/tda-witness/blob/main/examples/tda_witness_demo.ipynb)

In [ ]:
!pip install -q git+https://github.com/avacadobanana352/tda-witness.git#egg=tda-witness[all]

In [ ]:
import numpy as np
from tda import analyze, persistent_homology
from tda.datasets import make_circle, make_figure_eight, make_torus
from tda.visualization import (
    plot_complex, plot_filtration, plot_betti_summary,
    plot_persistence_diagram, plot_barcode,
)

## 1. Circle

A circle has one connected component and one loop: $\beta_0 = 1, \beta_1 = 1$.

In [ ]:
circle = make_circle(n_points=50, noise=0.05, seed=42)
result = analyze(circle, threshold=0.3, seed=42)
print("Betti numbers:", result["betti"])

In [ ]:
plot_complex(result["data"], result["landmarks"], result["graph"], result["complex"])

### Filtration

Sweep the threshold R and watch the complex grow. At low R you see many disconnected components. As R increases, edges appear and the loop forms.

In [ ]:
thresholds = np.linspace(0.05, 0.5, 25)
plot_filtration(circle, thresholds=thresholds, seed=42)

### Animated filtration GIF

Render the filtration as an animated GIF you can share.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection, LineCollection
from PIL import Image
from IPython.display import display, Image as IPImage
import io

anim_thresholds = np.linspace(0.02, 0.55, 40)
frames = []

for R in anim_thresholds:
    res = analyze(circle, threshold=float(R), simplex_dim=2, seed=42)
    lm_coords = res["data"][res["landmarks"]]
    graph = res["graph"]
    betti = res["betti"]
    while len(betti) < 3:
        betti.append(0)

    fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
    fig.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#1a1a2e")

    # triangles
    if len(res["complex"]) > 2:
        patches = [Polygon(lm_coords[tri], closed=True) for tri in res["complex"][2]]
        if patches:
            ax.add_collection(PatchCollection(patches, alpha=0.25, facecolor="#00d2ff", edgecolor="none"))

    # edges
    er, ec = np.where(np.triu(graph) == 1)
    if len(er) > 0:
        segs = [[lm_coords[r], lm_coords[c]] for r, c in zip(er, ec)]
        ax.add_collection(LineCollection(segs, colors="#5e60ce", linewidths=1.2, alpha=0.7))

    ax.scatter(res["data"][:, 0], res["data"][:, 1], s=20, c="#e0e0e0", zorder=5, edgecolors="none")
    ax.scatter(lm_coords[:, 0], lm_coords[:, 1], s=50, c="#ff6b6b", marker="D", zorder=6, edgecolors="white", linewidths=0.5)
    ax.set_title(f"R = {R:.3f}", fontsize=16, color="white", pad=10, fontweight="bold")
    ax.text(0.5, -0.06, f"B0={betti[0]}   B1={betti[1]}", transform=ax.transAxes, fontsize=14,
            color="#00d2ff", ha="center", fontfamily="monospace", fontweight="bold")
    ax.set(xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), aspect="equal", xticks=[], yticks=[])
    for spine in ax.spines.values():
        spine.set_visible(False)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    buf.seek(0)
    frames.append(Image.open(buf).copy())

# uniform size + save
max_w, max_h = max(f.width for f in frames), max(f.height for f in frames)
uniform = []
for f in frames:
    new = Image.new("RGBA", (max_w, max_h), (26, 26, 46, 255))
    new.paste(f, ((max_w - f.width) // 2, (max_h - f.height) // 2))
    uniform.append(new.convert("P", palette=Image.ADAPTIVE, colors=256))

durations = [120] * len(uniform)
durations[0], durations[-1] = 800, 2000

uniform[0].save("filtration.gif", save_all=True, append_images=uniform[1:], duration=durations, loop=0, optimize=True)
display(IPImage(filename="filtration.gif"))

### Betti numbers vs. threshold

In [ ]:
betti_seqs = [analyze(circle, threshold=float(r), seed=42)["betti"] for r in thresholds]
plot_betti_summary(thresholds, betti_seqs)

### Persistent homology

No threshold needed — persistence sweeps them all and tracks when features are born and die.

In [ ]:
ph = persistent_homology(circle, simplex_dim=2, seed=42)

# show only the significant pairs (lifetime > 0.1)
import math
for p in ph["pairs"]:
    lt = p["death"] - p["birth"] if not math.isinf(p["death"]) else float("inf")
    if lt > 0.1:
        death = "inf" if math.isinf(p["death"]) else f'{p["death"]:.3f}'
        print(f'  H{p["dim"]}: [{p["birth"]:.3f}, {death})  lifetime={lt if math.isinf(lt) else f"{lt:.3f}"}')

In [ ]:
plot_persistence_diagram(ph["pairs"])

In [ ]:
plot_barcode(ph["pairs"])

## 2. Figure eight

Two loops: $\beta_0 = 1, \beta_1 = 2$.

In [ ]:
fig8 = make_figure_eight(n_points=80, noise=0.03, seed=42)
result8 = analyze(fig8, threshold=0.25, seed=42)
print("Betti numbers:", result8["betti"])
plot_complex(result8["data"], result8["landmarks"], result8["graph"], result8["complex"])

## 3. Torus

A torus has $\beta_0 = 1, \beta_1 = 2, \beta_2 = 1$ — one component, two independent loops, one void.

In [ ]:
torus = make_torus(n_points=200, noise=0.02, seed=42)
result_t = analyze(torus, threshold=0.25, simplex_dim=2, seed=42)
print("Betti numbers:", result_t["betti"])
plot_complex(result_t["data"], result_t["landmarks"], result_t["graph"], result_t["complex"])

## 4. Bring your own data

Any `(n_points, n_dims)` numpy array works.

In [ ]:
# random point cloud
rng = np.random.default_rng(0)
my_data = rng.standard_normal((30, 2))

result_custom = analyze(my_data, threshold=0.8)
print("Betti numbers:", result_custom["betti"])
plot_complex(result_custom["data"], result_custom["landmarks"], result_custom["graph"], result_custom["complex"])